# ECON6083: Machine Learning in Economics
## Lecture 9 Exercise: Difference-in-Differences & RDD

**Coverage:** Lecture 9 — TWFE Crisis, Goodman-Bacon Decomposition, Callaway-Sant'Anna (2021), Sun-Abraham (2021), DML-DiD, and RDD Basics

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

## Part 1: Simulating Staggered Adoption Data

We simulate a panel with three treatment cohorts (early, mid, late) and never-treated units. Treatment effects are **heterogeneous by cohort** — this is the setting where TWFE fails.


In [ ]:
def simulate_staggered_did(n_units=500, n_periods=10, treatment_effect_type='heterogeneous'):
    """Simulate staggered adoption DiD data with heterogeneous treatment effects."""
    unit_fe = np.random.normal(0, 1, n_units)
    time_fe = np.cumsum(np.random.normal(0.05, 0.02, n_periods))
    cohorts = np.random.choice([2, 4, 6, np.inf], size=n_units, p=[0.3, 0.3, 0.2, 0.2])
    X1 = np.random.normal(0, 1, n_units)
    X2 = np.random.normal(0, 1, n_units)
    
    data = []
    for i in range(n_units):
        cohort = cohorts[i]
        for t in range(n_periods):
            treated = 1 if (t >= cohort and cohort != np.inf) else 0
            time_since = t - cohort if treated else 0
            Y0 = unit_fe[i] + time_fe[t] + 0.3 * X2[i] + np.random.normal(0, 0.5)
            
            if treatment_effect_type == 'heterogeneous':
                if treated:
                    if cohort == 2: tau = 3.0
                    elif cohort == 4: tau = 1.5
                    elif cohort == 6: tau = 0.5
                    else: tau = 0
                else: tau = 0
            else: tau = 0
            
            Y = Y0 + tau
            data.append({'unit_id': i, 'period': t, 'cohort': cohort, 'treat': treated,
                         'Y': Y, 'Y0': Y0, 'tau': tau, 'time_since': time_since,
                         'X1': X1[i], 'X2': X2[i]})
    return pd.DataFrame(data)

df = simulate_staggered_did(n_units=500, n_periods=10, treatment_effect_type='heterogeneous')

print(f"Units: {df['unit_id'].nunique()}, Periods: {df['period'].nunique()}, Total obs: {len(df)}")
print("\nCohort distribution:")
print(df.groupby('unit_id')['cohort'].first().value_counts())

### Q1.1 TWFE Estimation

Estimate the standard Two-Way Fixed Effects (TWFE) regression and compare to the true ATT.


In [ ]:
def estimate_twfe(df):
    """Estimate standard TWFE regression."""
    df_reg = df.copy()
    unit_dummies = pd.get_dummies(df_reg['unit_id'], prefix='unit')
    time_dummies = pd.get_dummies(df_reg['period'], prefix='period')
    X = pd.concat([unit_dummies, time_dummies, df_reg[['treat']]], axis=1)
    y = df_reg['Y']
    beta = np.linalg.lstsq(X.values.astype(np.float64), y.values.astype(np.float64), rcond=None)[0]
    return beta[____]  # TODO: extract the treatment coefficient (last element)

twfe_estimate = estimate_twfe(df)
true_att = df[df['treat'] == 1]['tau'].mean()
print(f"TWFE Estimate: {twfe_estimate:.3f}")
print(f"True ATT:      {true_att:.3f}")
print(f"TWFE Bias:     {abs(twfe_estimate - true_att):.3f}")

### Q1.2 Goodman-Bacon Decomposition

Implement the Goodman-Bacon decomposition to identify which 2×2 comparisons contribute to the TWFE estimate. Distinguish between **clean** comparisons and **forbidden** comparisons.

Hint: Loop over cohorts. For each cohort g:
1. Compare g vs never-treated (CLEAN)
2. Compare g vs later cohorts h>g, before h is treated (CLEAN)
3. Compare g vs earlier cohorts h<g, after h is treated (FORBIDDEN!)


In [ ]:
def goodman_bacon_weights(df):
    """Calculate Goodman-Bacon (2021) decomposition weights."""
    cohorts = sorted([c for c in df['cohort'].unique() if c != np.inf])
    never_treated = df[df['cohort'] == np.inf]['unit_id'].unique()
    comparisons = []
    
    for g in cohorts:
        g_units = df[df['cohort'] == g]['unit_id'].unique()
        
        # 1. Treated vs Never-treated (CLEAN)
        if len(never_treated) > 0:
            weight = len(g_units) * len(never_treated)
            comparisons.append({'type': 'Treated vs Never-treated', 'cohort_g': g, 'cohort_h': 'never',
                                'comparison': f'Cohort {int(g)} vs Never', 'weight': weight, 'valid': True})
        
        # 2. Early vs Late (before late is treated) - CLEAN
        for h in cohorts:
            if h > g:
                h_units = df[df['cohort'] == h]['unit_id'].unique()
                weight = len(g_units) * len(h_units)
                comparisons.append({'type': 'Early vs Late (clean)', 'cohort_g': g, 'cohort_h': h,
                                    'comparison': f'Cohort {int(g)} vs Cohort {int(h)} (pre-{int(h)})', 'weight': weight, 'valid': True})
        
        # 3. Late vs Early (after early is treated) - FORBIDDEN!
        for h in cohorts:
            if ____:  # TODO: condition for forbidden comparisons (already-treated earlier cohorts)
                h_units = df[df['cohort'] == h]['unit_id'].unique()
                weight = len(g_units) * len(h_units)
                comparisons.append({'type': 'Late vs Early (FORBIDDEN!)', 'cohort_g': g, 'cohort_h': h,
                                    'comparison': f'Cohort {int(g)} vs Cohort {int(h)} (post-{int(h)})', 'weight': weight, 'valid': False})
    
    comp_df = pd.DataFrame(comparisons)
    total_weight = comp_df['weight'].sum()
    comp_df['weight_share'] = comp_df['weight'] / total_weight
    return comp_df

comp_df = goodman_bacon_weights(df)
print('Comparison Types:')
print(comp_df.groupby(['type', 'valid'])['weight_share'].sum())

valid_weights = comp_df[comp_df['valid']]['weight_share'].sum()
invalid_weights = comp_df[~comp_df['valid']]['weight_share'].sum()
print(f"\nClean comparisons:     {valid_weights:.1%}")
print(f"Forbidden comparisons: {invalid_weights:.1%}")

### Q1.2b Verify: Re-estimate ATT Using Only Clean Comparisons

Goodman-Bacon shows that TWFE is a weighted average of all 2×2 DiD comparisons.
**Idea:** If we drop the forbidden comparisons and re-weight only the clean ones, do we recover the true ATT?

Steps:
1. For each **valid** (clean) comparison, compute its 2×2 DiD estimate
2. Re-weight by the Goodman-Bacon weights (restricted to clean comparisons only)
3. Compare to the true ATT


In [ ]:
def estimate_clean_att(df, comp_df):
    """Re-estimate ATT using only clean (valid) Goodman-Bacon comparisons."""
    clean_comps = comp_df[comp_df['valid']].copy()
    estimates = []
    weights = []
    
    for _, row in clean_comps.iterrows():
        g = row['cohort_g']
        h = row['cohort_h']
        
        # Define treatment and control units for this comparison
        if h == 'never':
            g_units = df[df['cohort'] == g]['unit_id'].unique()
            h_units = df[df['cohort'] == np.inf]['unit_id'].unique()
            # Compare post-g for g vs post-g for never-treated
            g_post = df[(df['unit_id'].isin(g_units)) & (df['period'] >= g)]['Y'].mean()
            g_pre = df[(df['unit_id'].isin(g_units)) & (df['period'] < g)]['Y'].mean()
            h_post = df[(df['unit_id'].isin(h_units)) & (df['period'] >= g)]['Y'].mean()
            h_pre = df[(df['unit_id'].isin(h_units)) & (df['period'] < g)]['Y'].mean()
        else:
            # Early (g) vs Late (h), before h is treated
            g_units = df[df['cohort'] == g]['unit_id'].unique()
            h_units = df[df['cohort'] == h]['unit_id'].unique()
            # Use periods [g, h-1] where g is treated and h is not yet treated
            g_post = df[(df['unit_id'].isin(g_units)) & (df['period'] >= g) & (df['period'] < h)]['Y'].mean()
            g_pre = df[(df['unit_id'].isin(g_units)) & (df['period'] < g)]['Y'].mean()
            h_post = df[(df['unit_id'].isin(h_units)) & (df['period'] >= g) & (df['period'] < h)]['Y'].mean()
            h_pre = df[(df['unit_id'].isin(h_units)) & (df['period'] < g)]['Y'].mean()
        
        if pd.notna(g_post) and pd.notna(g_pre) and pd.notna(h_post) and pd.notna(h_pre):
            did_estimate = (g_post - g_pre) - (h_post - h_pre)
            estimates.append(did_estimate)
            weights.append(row['weight'])
    
    if weights:
        clean_att = np.average(estimates, weights=weights)
    else:
        clean_att = np.nan
    
    return clean_att, estimates, weights

clean_att, est_list, w_list = estimate_clean_att(df, comp_df)
print(f"Clean-Weighted ATT: {clean_att:.3f}")
print(f"True ATT:           {true_att:.3f}")
print(f"TWFE Estimate:      {twfe_estimate:.3f}")
print(f"\nBias (Clean):       {abs(clean_att - true_att):.3f}")
print(f"Bias (TWFE):        {abs(twfe_estimate - true_att):.3f}")
print("\nConclusion: Removing forbidden comparisons eliminates the TWFE bias!")

### Q1.3 Callaway & Sant'Anna (2021) Group-Time ATT

Implement the CS2021 estimator for ATT(g,t). For a given cohort g and period t:
1. Identify treated units (cohort g) and control units (not-yet-treated by t)
2. Compute change from base period (g-1) for both groups
3. ATT(g,t) = (treated change) - (control change)


In [ ]:
def cs2021_att_gt(df, g, t, control_group='notyet'):
    """Estimate ATT(g,t) - Group-Time Average Treatment Effect on the Treated."""
    
    treated_units = df[df['cohort'] == g]['unit_id'].unique()
    if control_group == 'never':
        control_units = df[df['cohort'] == np.inf]['unit_id'].unique()
    else:
        control_units = df[(df['cohort'] > t) | (df['cohort'] == np.inf)]['unit_id'].unique()
    
    base_period = int(g) - 1
    treated_base = df[(df['unit_id'].isin(treated_units)) & (df['period'] == base_period)]['Y'].mean()
    treated_t = df[(df['unit_id'].isin(treated_units)) & (df['period'] == t)]['Y'].mean()
    delta_treated = ____ - treated_base  # TODO: compute treated change from base period to period t
    
    control_base = df[(df['unit_id'].isin(control_units)) & (df['period'] == base_period)]['Y'].mean()
    control_t = df[(df['unit_id'].isin(control_units)) & (df['period'] == t)]['Y'].mean()
    delta_control = ____ - control_base  # TODO: compute control change from base period to period t
    
    att_gt = ____  # TODO: compute group-time ATT as diff-in-diff
    
    return att_gt

# Estimate all ATT(g,t) for post-treatment periods
cohorts = sorted([c for c in df['cohort'].unique() if c != np.inf])
periods = sorted(df['period'].unique())

results = []
for g in cohorts:
    for t in periods:
        if t >= g:
            att = cs2021_att_gt(df, g, t)
            results.append({'cohort': int(g), 'period': t, 'event_time': int(t-g), 'ATT_gt': att})

att_gt_df = pd.DataFrame(results)
print('ATT(g,t) by cohort and event time:')
print(att_gt_df.pivot(index='cohort', columns='event_time', values='ATT_gt').round(2))

# Event-study aggregation
event_study = att_gt_df.groupby('event_time')['ATT_gt'].____  # TODO: aggregate ATT(g,t) by event time (mean)
print("\nEvent-study averages:")
print(event_study.round(2))

### Q1.4 Sun-Abraham (2021) Event Study

Implement the Sun-Abraham (2021) interactive weighted estimator.

Key idea: Estimate time FE using **never-treated only**, then compute cohort-specific deviations from these time effects, weighted by cohort size.


In [ ]:
def sa2021_event_study(df, max_event=4):
    """Sun & Abraham (2021) interactive weighted estimator."""
    never_treated = df[df['cohort'] == np.inf]['unit_id'].unique()
    never_data = df[df['unit_id'].isin(never_treated)]
    time_fe = never_data.groupby('period')['Y'].mean()
    
    cohorts = sorted([c for c in df['cohort'].unique() if c != np.inf])
    sa_results = []
    
    for e in range(-3, max_event + 1):
        weighted_effects = []
        weights = []
        
        for g in cohorts:
            t = int(g) + e
            if t < 0 or t >= df['period'].max():
                continue
            cohort_units = df[df['cohort'] == g]['unit_id'].unique()
            n_g = len(cohort_units)
            base_period = int(g) - 1
            if base_period < 0:
                continue
            
            y_g_base = df[(df['unit_id'].isin(cohort_units)) & (df['period'] == base_period)]['Y'].mean()
            y_g_t = df[(df['unit_id'].isin(cohort_units)) & (df['period'] == t)]['Y'].mean()
            time_change = time_fe[t] - time_fe[base_period]
            cohort_effect = (y_g_t - y_g_base) - time_change
            
            weighted_effects.append(cohort_effect)
            weights.append(n_g)
        
        if weights:
            beta_e = np.average(weighted_effects, weights=weights)
            sa_results.append({'event_time': e, 'beta_e': beta_e, 'n_cohorts': len(weights)})
    
    return pd.DataFrame(sa_results)

sa_results = sa2021_event_study(df, max_event=4)
print(sa_results.round(3))

# Plot SA2021 event study
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if x < 0 else 'blue' for x in sa_results['event_time']]
ax.bar(sa_results['event_time'], sa_results['beta_e'], color=colors, alpha=0.6, edgecolor='black')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axvline(x=-0.5, color='gray', linestyle='--')
ax.set_xlabel('Event Time (years relative to treatment)')
ax.set_ylabel('ATT')
ax.set_title("Sun & Abraham (2021) Event Study Estimates")
plt.show()

### Q1.4b Modern DiD Meets ML: CS2021 with ML-Based Control Outcomes

The CS2021 estimator uses simple mean differences for control outcomes. But if we have high-dimensional covariates $X$, we can use ML to better model $E[Y(0) \mid X]$ for the control group.

**Idea:** Replace the simple control mean in CS2021 with a Random Forest prediction of $Y(0)$ based on covariates.

Steps:
1. Train a Random Forest on never-treated units to predict $Y$ from covariates $X$
2. For each treated observation, predict its counterfactual $\hat{Y}(0)$ using the RF
3. ATT(g,t) = Average of $(Y - \hat{Y}(0))$ for cohort $g$ at time $t$
4. Compare to naive CS2021 (simple mean) — do we reduce variance?


In [ ]:
def cs2021_att_gt_ml(df, g, t, control_group='notyet'):
    """CS2021 ATT(g,t) with ML-based control outcome model (Random Forest)."""
    from sklearn.ensemble import RandomForestRegressor
    
    treated_units = df[df['cohort'] == g]['unit_id'].unique()
    if control_group == 'never':
        control_units = df[df['cohort'] == np.inf]['unit_id'].unique()
    else:
        control_units = df[(df['cohort'] > t) | (df['cohort'] == np.inf)]['unit_id'].unique()
    
    base_period = int(g) - 1
    
    # Get data for ML model training (control units only, base period)
    cov_cols = [c for c in df.columns if c.startswith('X')]
    control_data = df[(df['unit_id'].isin(control_units)) & (df['period'] == base_period)]
    
    if len(control_data) < 10 or len(cov_cols) == 0:
        # Fall back to naive CS2021
        return cs2021_att_gt(df, g, t, control_group)
    
    # Train Random Forest on control units to predict Y from X
    rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
    rf.fit(control_data[cov_cols], control_data['Y'])
    
    # For treated units at time t: predict counterfactual Y(0) using RF
    treated_data_t = df[(df['unit_id'].isin(treated_units)) & (df['period'] == t)]
    treated_data_base = df[(df['unit_id'].isin(treated_units)) & (df['period'] == base_period)]
    
    # ML-predicted counterfactuals at time t (using base period X for simplicity)
    X_treated_base = treated_data_base[cov_cols]
    Y1_t = treated_data_t['Y'].mean()  # Observed outcome
    Y0_hat_t = rf.predict(X_treated_base).mean()  # Predicted counterfactual
    
    # Similarly for controls
    control_data_t = df[(df['unit_id'].isin(control_units)) & (df['period'] == t)]
    X_control_base = df[(df['unit_id'].isin(control_units)) & (df['period'] == base_period)][cov_cols]
    Y0_hat_control_t = rf.predict(X_control_base).mean() if len(X_control_base) > 0 else Y0_hat_t
    
    # ATT(g,t) = (Y1_t - Y0_hat_t) - (Y0_control_t - Y0_hat_control_t)
    # Simplified: just use ML-imputed DiD
    att_gt_ml = (Y1_t - Y0_hat_t) - (control_data_t['Y'].mean() - Y0_hat_control_t)
    
    return att_gt_ml

# Compare naive vs ML-based CS2021
results_comparison = []
for g in cohorts[:2]:  # Just first 2 cohorts for speed
    for t in [int(g), int(g)+1]:  # Post-treatment periods
        att_naive = cs2021_att_gt(df, g, t)
        att_ml = cs2021_att_gt_ml(df, g, t)
        results_comparison.append({
            'cohort': int(g), 'period': t, 'event_time': t-int(g),
            'ATT_naive': att_naive, 'ATT_ML': att_ml, 'difference': att_ml - att_naive
        })

comp_df_ml = pd.DataFrame(results_comparison)
print("CS2021: Naive vs ML-Based Control Outcomes")
print(comp_df_ml.round(3))

# Overall comparison
print(f"\nMean |ATT_ML - ATT_naive|: {comp_df_ml['difference'].abs().mean():.3f}")
print("Note: ML can improve efficiency by using covariate information in the control model.")

#### Discussion: When Does ML Help in DiD?

You may find that ML and naive estimates are similar in this simulation. **This is expected** — ML is not always better. Let’s understand why:

**When ML helps:**
1. **High-dimensional covariates**: Many $X$ variables that affect outcomes (curse of dimensionality)
2. **Nonlinear relationships**: $E[Y(0) \mid X]$ is complex (polynomials, interactions, thresholds)
3. **Selection on observables is complex**: Propensity scores depend nonlinearly on many covariates
4. **Small treatment groups**: Simple means have high variance; ML borrows strength across units

**When ML adds little value:**
1. **Few covariates with linear effects**: OLS / simple means work fine
2. **Large samples with good overlap**: Law of large numbers makes simple means consistent
3. **No confounding**: If treatment is randomly assigned, we don’t need sophisticated control models

**Key insight**: ML is a tool for modeling $E[Y(0) \mid X]$ and $P(D \mid X)$. If these are simple, ML is overkill. If they are complex, ML is essential. The DML framework ensures valid inference **regardless** of whether ML improves precision — the orthogonality property guarantees we don’t introduce bias.


### Q1.5 DML-DiD with High-Dimensional Controls

Implement a simplified DML-DiD estimator. Use cross-fitting and IPW with logistic regression for propensity scores.

Steps:
1. Split data into K folds
2. For each fold, train propensity score model on other folds
3. Compute IPW ATT using estimated propensity scores


In [ ]:
def dml_did(df, n_splits=3):
    """
    DML-DiD estimation with high-dimensional controls (Chang 2020).
    
    Key insight: Use DiD (change from base period) + orthogonal score
    to handle high-dimensional covariates with ML.
    """
    cov_cols = [c for c in df.columns if c.startswith('X')]
    cohorts = sorted([c for c in df['cohort'].unique() if c != np.inf])
    all_att_estimates = []
    
    for g in cohorts:
        base_period = int(g) - 1
        
        for t in range(int(g), 10):  # Post-treatment periods
            treated_units = df[df['cohort'] == g]['unit_id'].unique()
            control_units = df[df['cohort'] == np.inf]['unit_id'].unique()
            
            # Build dataset: delta_Y = Y_t - Y_{g-1} for each unit
            delta_data = []
            for uid in np.concatenate([treated_units, control_units]):
                y_t = df[(df['unit_id'] == uid) & (df['period'] == t)]['Y'].values
                y_base = df[(df['unit_id'] == uid) & (df['period'] == base_period)]['Y'].values
                if len(y_t) > 0 and len(y_base) > 0:
                    delta_y = y_t[0] - y_base[0]
                    d = 1 if uid in treated_units else 0
                    x_vals = df[(df['unit_id'] == uid) & (df['period'] == base_period)][cov_cols].values
                    if len(x_vals) > 0:
                        delta_data.append({
                            'unit_id': uid, 'delta_Y': delta_y, 'D': d,
                            **{cov_cols[i]: x_vals[0][i] for i in range(len(cov_cols))}
                        })
            
            if len(delta_data) < 20:
                continue
            
            delta_df = pd.DataFrame(delta_data)
            
            # Cross-fitting: train on K-1 folds, predict on hold-out fold
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            fold_scores = []
            
            for train_idx, test_idx in kf.split(delta_df):
                train = delta_df.iloc[train_idx]
                test = delta_df.iloc[test_idx]
                
                # Step 1: Estimate propensity score g(X) = P(D=1|X)
                g_model = LogisticRegression(max_iter=200, C=1.0, solver='lbfgs')
                g_model.fit(train[cov_cols].values, train['D'].values)
                g_hat = np.clip(g_model.predict_proba(test[cov_cols].values)[:, 1], 0.05, 0.95)
                
                # Step 2: Estimate outcome model l(X) = E[delta_Y|X,D=0] (on controls only)
                train_control = train[train['D'] == 0]
                if len(train_control) < 5:
                    continue
                l_model = RandomForestRegressor(n_estimators=30, max_depth=5, random_state=42)
                l_model.fit(train_control[cov_cols].values, train_control['delta_Y'].values)
                l_hat = l_model.predict(test[cov_cols].values)
                
                # Step 3: Compute Chang (2020) orthogonal score
                D = test['D'].values
                delta_Y = test['delta_Y'].values
                
                # TODO: Fill in the orthogonal score formula
                # psi_i = (D_i - g_hat_i) * (delta_Y_i - l_hat_i) / (g_hat_i * (1 - g_hat_i))
                psi = ____  # Hint: Use D, g_hat, delta_Y, l_hat
                att_dr = np.mean(psi)
                fold_scores.append(att_dr)
            
            if fold_scores:
                all_att_estimates.append(np.mean(fold_scores))
    
    return np.mean(all_att_estimates) if all_att_estimates else np.nan

dml_estimate = dml_did(df)
print(f"DML-DiD Estimate: {dml_estimate:.3f}")
print(f"True ATT:         {true_att:.3f}")
print(f"DML-DiD Bias:     {abs(dml_estimate - true_att):.3f}")

### Q1.6 Summary Comparison

Compile results into a summary table comparing TWFE, CS2021, SA2021, and DML-DiD.


In [ ]:
cs_overall = att_gt_df['ATT_gt'].____  # TODO: compute overall CS2021 ATT estimate (average over all ATT(g,t))
sa_overall = sa_results[sa_results['event_time'] >= 0]['beta_e'].____  # TODO: compute overall SA2021 post-treatment average

results_summary = pd.DataFrame({
    'Method': ['True ATT', 'TWFE', 'CS2021', 'SA2021', 'DML-DiD'],
    'Estimate': [true_att, twfe_estimate, ____, ____, dml_estimate],  # TODO: insert CS and SA overall estimates
    'Bias': [0, abs(twfe_estimate - true_att), abs(cs_overall - true_att),
             abs(sa_overall - true_att), abs(dml_estimate - true_att)]
})

print(results_summary.to_string(index=False))

## Part 2: Regression Discontinuity Design (RDD) Meets ML

RDD exploits deterministic treatment assignment rules based on a running variable (e.g., scholarship if GPA $\geq$ 3.5). Traditional RDD only uses data near the cutoff. But when confounders are high-dimensional or nonlinear, ML can help by modeling the outcome-covariate relationship more flexibly.


### Q2.1 Simulate RDD Data with High-Dimensional Confounding

Generate data where enrollment probability depends on GPA, high-dimensional covariates ($X_1, X_2, X_3$), and a scholarship cutoff at GPA = 3.5.


In [ ]:
# Generate RDD data with complex confounding
np.random.seed(42)
n = 1000

# Running variable (GPA)
gpa = np.random.uniform(2.5, 4.0, n)
cutoff = 3.5

# High-dimensional covariates
X1 = np.random.normal(0, 1, n)  # Family income proxy
X2 = np.random.normal(0, 1, n)  # School quality
X3 = np.random.normal(0, 1, n)  # Urban/rural

# GPA correlated with covariates
gpa = gpa + 0.1*X1 + 0.05*X2
gpa = np.clip(gpa, 2.5, 4.0)

# Treatment: scholarship if GPA >= cutoff
treatment = (gpa >= cutoff).astype(int)

# TRUE causal effect
true_rdd_effect = 0.25

# Complex outcome with nonlinearity and interactions
enroll_prob = (0.2 + 
               0.3*(gpa - 2.5) +           # Linear GPA trend
               0.1*(gpa - 2.5)**2 +        # Quadratic trend
               0.15*X1 +                   # Income effect
               0.1*X2*(gpa - 2.5) +       # Interaction
               0.05*X3 +                   # Urban/rural
               true_rdd_effect * treatment +  # CAUSAL EFFECT
               np.random.normal(0, 0.08, n))

enroll_prob = np.clip(enroll_prob, 0, 1)
enroll = (np.random.uniform(0, 1, n) < enroll_prob).astype(int)

rdd_df = pd.DataFrame({
    'gpa': gpa,
    'treatment': treatment,
    'enroll': enroll,
    'X1': X1,
    'X2': X2,
    'X3': X3
})

print(f"True effect: {true_rdd_effect:.3f}")

### Q2.2 Naïve RDD (Baseline)

Estimate the treatment effect as a simple difference in means within a bandwidth $h$ of the cutoff.


In [ ]:
# Naïve RDD: simple difference in means within bandwidth
bandwidth = 0.2
near = rdd_df[abs(rdd_df['gpa'] - cutoff) <= bandwidth]

# Your code here: compute simple difference
# naive_est = ...

print(f"Naïve RDD (h={bandwidth}): {naive_est:.3f}")
print(f"Bias: {abs(naive_est - true_rdd_effect):.3f}")

### Q2.3 Local Linear RDD (Standard Approach)

Fit separate linear regressions on each side of the cutoff and compare the intercepts.


In [ ]:
# Local linear RDD implementation
def local_linear_rdd(df, running, outcome, cutoff, h):
    near = df[abs(df[running] - cutoff) <= h].copy()
    near['dist'] = near[running] - cutoff
    
    # Below cutoff: fit Y = beta0 + beta1 * dist
    below = near[near[running] < cutoff]
    X_b = np.column_stack([np.ones(len(below)), below['dist']])
    y_b = below[outcome].values
    beta_b = np.linalg.lstsq(X_b, y_b, rcond=None)[0]
    y0_hat = beta_b[0]  # intercept at cutoff
    
    # Above cutoff: same approach
    above = near[near[running] >= cutoff]
    X_a = np.column_stack([np.ones(len(above)), above['dist']])
    y_a = above[outcome].values
    beta_a = np.linalg.lstsq(X_a, y_a, rcond=None)[0]
    y1_hat = beta_a[0]  # intercept at cutoff
    
    return y1_hat - y0_hat

ll_est = local_linear_rdd(rdd_df, 'gpa', 'enroll', cutoff, bandwidth)
print(f"Local Linear (h={bandwidth}): {ll_est:.3f}")
print(f"Bias: {abs(ll_est - true_rdd_effect):.3f}")

### Q2.4 ML-Augmented RDD (ML Meets RDD!)

**Key Idea**: Use ML to model $E[Y | X, GPA, D=0]$ on all controls, predict counterfactuals, then do RDD on residualized outcome $(Y - \hat{Y})$.

**Why this works**: ML borrows strength from all controls (not just near the cutoff) to adjust for confounding.


In [ ]:
# ML-RDD implementation
from sklearn.ensemble import RandomForestRegressor

def ml_rdd(df, running, outcome, cov_cols, cutoff, h):
    near = df[abs(df[running] - cutoff) <= h].copy()
    
    # Step 1: Train Random Forest on controls (D=0)
    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    control_data = near[near[running] < cutoff]
    
    # Fit on controls using GPA + covariates
    X_train = control_data[[running] + cov_cols].values
    y_train = control_data[outcome].values
    model.fit(X_train, y_train)
    
    # Step 2: Predict counterfactual for everyone
    near['Y_pred'] = model.predict(near[[running] + cov_cols].values)
    near['Y_resid'] = near[outcome] - near['Y_pred']
    
    # Step 3: Local linear RDD on residualized outcome
    near['dist'] = near[running] - cutoff
    below = near[near[running] < cutoff]
    X_b = np.column_stack([np.ones(len(below)), below['dist']])
    beta_b = np.linalg.lstsq(X_b, below['Y_resid'].values, rcond=None)[0]
    
    above = near[near[running] >= cutoff]
    X_a = np.column_stack([np.ones(len(above)), above['dist']])
    beta_a = np.linalg.lstsq(X_a, above['Y_resid'].values, rcond=None)[0]
    
    return beta_a[0] - beta_b[0], near

cov_cols = ['X1', 'X2', 'X3']
ml_est, _ = ml_rdd(rdd_df, 'gpa', 'enroll', cov_cols, cutoff, bandwidth)
print(f"ML-RDD: {ml_est:.3f}, Bias: {abs(ml_est - true_rdd_effect):.3f}")

### Q2.5 Comparison and Bandwidth Sensitivity

Compare all methods and test sensitivity to bandwidth choice.


In [ ]:
# TODO: Comparison table and bandwidth sensitivity
print("\n=== COMPARISON ===")
print(f"{'Method':<25} {'Estimate':>10} {'Bias':>10}")
print("-" * 50)
print(f"{'True Effect':<25} {true_rdd_effect:>10.3f} {0.0:>10.3f}")
print(f"{'Naive RDD':<25} {naive_est:>10.3f} {abs(naive_est - true_rdd_effect):>10.3f}")
print(f"{'Local Linear':<25} {ll_est:>10.3f} {abs(ll_est - true_rdd_effect):>10.3f}")
print(f"{'ML-RDD (RF)':<25} {ml_est:>10.3f} {abs(ml_est - true_rdd_effect):>10.3f}")

# Bandwidth sensitivity
print("\n=== BANDWIDTH SENSITIVITY ===")
bandwidths = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
results = []
for h in bandwidths:
    ll = local_linear_rdd(rdd_df, 'gpa', 'enroll', cutoff, h) if len(
        rdd_df[abs(rdd_df['gpa']-cutoff) <= h]) > 50 else np.nan
    ml, _ = ml_rdd(rdd_df, 'gpa', 'enroll', cov_cols, cutoff, h, 'rf')
    results.append({'h': h, 'LL': ll, 'ML': ml})

for r in results:
    print(f"h={r['h']:.2f}: Local Linear={r['LL']:.3f}, ML-RDD={r['ML']:.3f}")

print(
    f"\nStd across bandwidths - LL: {pd.DataFrame(results)['LL'].std():.3f}, ML: {pd.DataFrame(results)['ML'].std():.3f}")

### Q2.6 Discussion: When Does ML Help in RDD?

Write 2-3 sentences explaining why ML-RDD outperforms traditional methods in this setting, and when it might **not** help.

**Your answer:** *(write here)*

---
## Part 3: Synthetic Control Placebo Test

Synthetic Control (SCM) constructs a weighted combination of control units to match the treated unit's pre-treatment trajectory.

**The key robustness check:** Run SCM for *every* control unit as if it were treated. If the true effect is an outlier among all placebo effects, the estimate is credible.

This is the **permutation test** proposed by Abadie, Diamond & Hainmueller (2010).

### Q3.1 Simulating Panel Data for SCM

Simulate one treated unit and 20 control units over 30 periods (20 pre-treatment, 10 post-treatment).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)

T_pre = 20
T_post = 10
T = T_pre + T_post
J = 20
time = np.arange(T)

# Generate donor pool
Y_donor = np.zeros((J, T))
for j in range(J):
    trend = 0.5 * time + np.random.normal(0, 0.3)
    unit_fe = np.random.normal(5, 2)
    Y_donor[j, :] = unit_fe + trend + np.random.normal(0, 1, T)

# Generate treated unit with post-treatment effect
true_effect = np.concatenate([np.zeros(T_pre),
                              2.0 * np.log1p(np.arange(1, T_post + 1))])
Y_treated = 7 + 0.5 * time + true_effect + np.random.normal(0, 0.8, T)


### Q3.2 Estimate Synthetic Control Weights

Write a function that estimates SCM weights by minimizing pre-treatment SSE, subject to non-negativity and sum-to-one constraints.

In [ ]:
def sc_weights(y_treated_pre, y_donor_pre):
    """Estimate SC weights: min pre-treatment SSE, sum(w)=1, w>=0."""
    n_units = y_donor_pre.shape[0]
    def objective(w):
        return np.sum((y_treated_pre - y_donor_pre.T @ w) ** 2)
    cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = [(0, 1) for _ in range(n_units)]
    w0 = np.ones(n_units) / n_units
    res = minimize(objective, w0, method='SLSQP', bounds=bounds, constraints=cons)
    return res.x

# True SCM effect
w_true = sc_weights(Y_treated[:T_pre], Y_donor[:, :T_pre])
Y_sc_true = Y_donor.T @ w_true
effect_true = Y_treated - Y_sc_true


### Q3.3 Run Placebo Tests for Every Control Unit

For each donor unit $j$, exclude it from the donor pool, estimate SCM using the remaining units, and compute the placebo gap.

In [ ]:
effect_placebo = np.zeros((J, T))
for j in range(J):
    other_units = [k for k in range(J) if k != j]
    Y_pool = Y_donor[other_units, :]
    w_placebo = sc_weights(Y_donor[j, :T_pre], Y_pool[:, :T_pre])
    Y_sc_placebo = Y_pool.T @ w_placebo
    effect_placebo[j, :] = Y_donor[j, :] - Y_sc_placebo

# Compute post/pre RMSPE ratios
def rmspe(effect, start, end):
    return np.sqrt(np.mean(effect[start:end] ** 2))

ratio_true = rmspe(effect_true, T_pre, T) / rmspe(effect_true, 0, T_pre)
ratios_placebo = np.array([rmspe(effect_placebo[j], T_pre, T) / rmspe(effect_placebo[j], 0, T_pre)
                           for j in range(J)])
p_value = np.mean(ratios_placebo >= ratio_true)
print(f'True RMSPE ratio: {ratio_true:.3f}')
print(f'Placebo p-value: {p_value:.3f}')


### Q3.4 Visualize the Placebo Test

Create a two-panel figure:
- **Left:** Gap over time for true unit (red) vs all placebo units (gray)
- **Right:** Histogram of placebo RMSPE ratios with true unit marked (red line)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Time series
ax1 = axes[0]
for j in range(J):
    ax1.plot(time, effect_placebo[j, :], color='lightgray', linewidth=0.8, alpha=0.7, zorder=1)
ax1.plot(time, effect_true, color='#C41E3A', linewidth=2.8, zorder=3, label='Treated unit')
ax1.axvline(x=T_pre - 0.5, color='black', linestyle='--', linewidth=1.2, zorder=2)
ax1.axhline(y=0, color='black', linewidth=0.5, zorder=2)
ax1.set_xlabel('Time Period')
ax1.set_ylabel('Gap (Treated - Synthetic Control)')
ax1.set_title('Placebo Test: Gap in Outcome')
ax1.legend(loc='upper left')

# Right: Ratio distribution
ax2 = axes[1]
n_bins, bins, patches = ax2.hist(ratios_placebo, bins=12, color='steelblue',
                                 edgecolor='white', alpha=0.7, zorder=1)
ax2.axvline(x=ratio_true, color='#C41E3A', linewidth=2.8, zorder=3,
            label=f'True unit (ratio={ratio_true:.2f})')
for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge >= ratio_true:
        patch.set_facecolor('#C41E3A')
        patch.set_alpha(0.4)
ax2.set_xlabel('Post-period RMSPE / Pre-period RMSPE')
ax2.set_ylabel('Number of Control Units')
ax2.set_title(f'Ratio Distribution (p-value = {p_value:.2f})')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


### Q3.5 Interpretation

**Questions:**
1. What does the p-value tell us about the credibility of the SCM estimate?
2. Why do we compare *ratios* of post- vs pre-treatment RMSPE instead of raw post-treatment gaps?
3. What would it mean if many placebo units had ratios similar to or larger than the true unit?

---\n
\n
## Part 4: Interpretation\n
\n
Answer the following questions briefly.\n

**Q3.1 Why does TWFE fail under staggered adoption with heterogeneous effects?**\n
\n
TWFE fails because it estimates a weighted average of all 2x2 DiD comparisons, where some comparisons use already-treated units as controls. When treatment effects are heterogeneous across cohorts and time, these 'forbidden comparisons' (Goodman-Bacon 2021) produce negative weights, causing bias. TWFE essentially compares late adopters' post-treatment outcomes to early adopters' post-treatment outcomes—both are treated, so this doesn't identify the ATT.

**Q3.2 What is the key innovation of CS2021 compared to TWFE?**\n
\n
CS2021 (Callaway & Sant'Anna 2021) only uses never-treated and not-yet-treated units as controls, avoiding the negative weighting problem. They compute cohort-specific ATT(g,t) for each treatment group g at each time t, then aggregate using user-specified weights. This ensures all comparisons are valid DiD contrasts with proper counterfactuals.

**Q3.3 How does Sun-Abraham (2021) differ from Callaway-Sant'Anna in estimating event-study dynamic effects?**\n
\n
SA2021 uses an interacted weighted estimator that directly parameterizes relative event time (e periods before/after treatment). They interact cohort indicators with event-time dummies, then re-center to ensure the comparison group is never-treated units. CS2021 computes cohort-time-specific effects first, then aggregates; SA2021 estimates dynamic effects directly via the interaction structure.

**Q3.4 What is the bias-variance trade-off in RDD bandwidth selection?**\n
\n
**Small bandwidth**: Lower bias (units near cutoff are more comparable), but higher variance (fewer observations).\n
**Large bandwidth**: Lower variance (more data), but higher bias (units far from cutoff may differ systematically).\n
Optimal bandwidth balances MSE = Bias² + Variance using data-driven selectors (IK, CCT).

**Q3.5 When would you prefer DML-DiD over CS2021?**\n
\n
Use **DML-DiD** when: (1) high-dimensional covariates with complex selection, (2) you need valid inference under weak assumptions, (3) single treated cohort setting.\n
Use **CS2021** when: (1) staggered adoption with multiple cohorts, (2) treatment timing is main variation, (3) you want transparent aggregation. They're complements: CS2021 handles timing; DML handles confounding.